# Explosiveness Transfers, Smoothness Misleads
### Rebuilding the Movement Efficiency Index (MEI) with Combine sensor tracking

**NFL Big Data Bowl 2027: reproducible analysis notebook.**

This notebook rebuilds every number and figure in the writeup from the raw competition data:

1. Movement features and the Movement Efficiency Index (MEI) from 10 Hz Combine tracking
2. Regular-season NFL outcomes, including DB reaction metrics from game tracking
3. Leave-one-draft-class-out models with bootstrap CIs and permutation tests
4. The DL transfer test, learned MEI weights, reliability and validation tables
5. The writeup figures

Runtime is about 20 minutes on a Kaggle CPU session, most of it the 200-permutation model run.

In [ ]:
import os
from pathlib import Path

# locate the competition data wherever Kaggle mounts it
if 'BDB_RAW' not in os.environ:
    hits = sorted(Path('/kaggle/input').rglob('combine_results.csv'))
    if not hits:
        raise FileNotFoundError('Add the nfl-big-data-bowl-2027 competition data to this notebook')
    os.environ['BDB_RAW'] = str(hits[0].parent)
os.environ.setdefault('BDB_PROC', 'processed')
os.environ.setdefault('BDB_FIG', 'figures')
for d in ['src/features', 'tests', os.environ['BDB_PROC'], os.environ['BDB_FIG']]:
    Path(d).mkdir(parents=True, exist_ok=True)
print({k: os.environ[k] for k in ['BDB_RAW', 'BDB_PROC', 'BDB_FIG']})

## Source code
The analysis is a small package, written out here so the notebook is self-contained.

In [ ]:
%%writefile src/__init__.py
"""Combine tracking to NFL performance analysis."""


In [ ]:
%%writefile src/paths.py
"""Data and output locations. Override with BDB_RAW / BDB_PROC / BDB_FIG (e.g. on Kaggle)."""
import os
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
RAW = Path(os.environ.get("BDB_RAW", ROOT / "data/raw/nfl-big-data-bowl-2027"))
PROC = Path(os.environ.get("BDB_PROC", ROOT / "data/processed"))
FIG = Path(os.environ.get("BDB_FIG", ROOT / "figures"))


In [ ]:
%%writefile src/features/__init__.py
"""Combine movement features."""


In [ ]:
%%writefile src/features/kinematics.py
"""Per-attempt movement signals and MEI components from 10 Hz tracking.

Units: yards, seconds. Heading is computed from smoothed x/y velocity
(counter-clockwise positive), so a positive heading change is a left turn.
"""
from dataclasses import dataclass, field

import numpy as np
from scipy.signal import find_peaks, savgol_filter

DT = 0.1
ACTIVE_SPEED = 0.5      # yd/s: movement onset / end of activity
CUT_WINDOW = 8          # frames each side used for direction of travel
CUT_MIN_TURN = 35.0     # deg: minimum heading change to count as a cut
CUT_MIN_SPEED = 1.0     # yd/s: mean velocity needed on both sides of a cut
CUT_SPACING = 8         # frames: minimum gap between two cuts
APPROACH = 15           # frames: look-back for approach speed
DECEL_THRESH = -1.0     # yd/s^2: deceleration episode threshold
DECEL_MIN_DV = 1.5      # yd/s: minimum speed drop for an episode
RECOVER_FRAC = 0.9      # share of pre-cut speed that counts as recovered


@dataclass
class Signals:
    t: np.ndarray
    v: np.ndarray        # smoothed speed
    at: np.ndarray       # signed tangential acceleration
    at_resid: np.ndarray  # acceleration minus its smooth trend
    vx: np.ndarray
    vy: np.ndarray


@dataclass
class Cut:
    turn: float          # signed heading change, degrees (+ = left)
    v_pre: float         # approach speed
    v_min: float         # slowest speed through the cut
    elapsed: float       # slowdown start -> recovery
    dist: float          # distance covered over `elapsed`
    time_lost: float
    cod_eff: float
    recovery_acc: float


@dataclass
class AttemptFeatures:
    peak_speed: float = np.nan
    accel_eff: float = np.nan
    decel_ctrl: float = np.nan
    smoothness: float = np.nan
    cuts: list = field(default_factory=list)


def _savgol(x, window, deriv=0):
    window = min(window, len(x) - (1 - len(x) % 2))
    if window < 5:
        return np.gradient(x, DT) if deriv else x.astype(float)
    return savgol_filter(x, window, 2, deriv=deriv, delta=DT)


def signals(x, y, s):
    """Trim to active movement and derive smoothed kinematic signals."""
    x, y, s = map(np.asarray, (x, y, s))
    moving = np.flatnonzero(s > ACTIVE_SPEED)
    if len(moving) < 5:
        return None
    lo, hi = max(moving[0] - 2, 0), min(moving[-1] + 3, len(s))
    x, y, s = x[lo:hi], y[lo:hi], s[lo:hi]
    v = np.clip(_savgol(s, 7), 0, None)
    at = _savgol(s, 7, deriv=1)
    at_resid = _savgol(s, 5, deriv=1) - _savgol(s, 15, deriv=1)
    vx, vy = _savgol(x, 7, deriv=1), _savgol(y, 7, deriv=1)
    return Signals(np.arange(len(v)) * DT, v, at, at_resid, vx, vy)


def _runs(mask):
    """(start, end_exclusive) index pairs of True runs."""
    d = np.diff(np.r_[0, mask.astype(int), 0])
    return list(zip(np.flatnonzero(d == 1), np.flatnonzero(d == -1)))


def acceleration_efficiency(sg):
    """0.9 x peak speed / time from onset to reaching 0.9 x peak speed."""
    vmax = sg.v.max()
    onset = np.argmax(sg.v > ACTIVE_SPEED)
    reach = onset + np.argmax(sg.v[onset:] >= 0.9 * vmax)
    t = max(sg.t[reach] - sg.t[onset], DT)
    return vmax, 0.9 * vmax / t


def deceleration_control(sg):
    """Largest deceleration episode: speed lost / distance travelled (1/s)."""
    best = None
    for a, b in _runs(sg.at < DECEL_THRESH):
        b = min(b, len(sg.v) - 1)
        dv = sg.v[a] - sg.v[b]
        if dv >= DECEL_MIN_DV:
            dist = np.trapezoid(sg.v[a:b + 1], dx=DT)
            if best is None or dv > best[0]:
                best = (dv, dv / max(dist, 0.05))
    return np.nan if best is None else best[1]


def smoothness(sg):
    """Negative log RMS of acceleration around its trend (higher = smoother)."""
    return -np.log(np.sqrt(np.mean(sg.at_resid ** 2)) + 1e-6)


def turn_profile(sg):
    """Signed angle (deg, + = left) between mean velocity before and after each frame.

    Comparing whole windows rather than instantaneous heading keeps plant-and-reverse
    cuts, where speed passes through zero, and rounded cuts on the same footing.
    """
    k, n = CUT_WINDOW, len(sg.v)
    ang = np.zeros(n)
    for i in range(k, n - k):
        ax, ay = sg.vx[i - k:i].mean(), sg.vy[i - k:i].mean()
        bx, by = sg.vx[i + 1:i + k + 1].mean(), sg.vy[i + 1:i + k + 1].mean()
        if np.hypot(ax, ay) >= CUT_MIN_SPEED and np.hypot(bx, by) >= CUT_MIN_SPEED:
            ang[i] = np.degrees(np.arctan2(ax * by - ay * bx, ax * bx + ay * by))
    return ang


def cuts(sg):
    """Detect changes of direction and score each one."""
    out = []
    ang = turn_profile(sg)
    peaks, _ = find_peaks(np.abs(ang), height=CUT_MIN_TURN, distance=CUT_SPACING)
    n = len(sg.v)
    for j, i in enumerate(peaks):
        lo = max(i - APPROACH, 0)
        nxt = peaks[j + 1] if j + 1 < len(peaks) else n
        v_pre = sg.v[lo:i + 1].max()
        # slowdown starts at the last frame before the cut still near approach speed
        near = np.flatnonzero(sg.v[lo:i + 1] >= RECOVER_FRAC * v_pre)
        a = lo + near[-1]
        i_min = a + int(np.argmin(sg.v[a:min(i + CUT_WINDOW, n - 1) + 1]))
        # recovered = regains 90% of approach speed; on short legs the next cut can come
        # first, in which case recovery ends at the fastest frame before that cut
        seg = sg.v[i_min:max(nxt, i_min + 1)]
        after = np.flatnonzero(seg >= RECOVER_FRAC * v_pre)
        i_rec = i_min + (after[0] if len(after) else int(np.argmax(seg)))
        dist = np.trapezoid(sg.v[a:i_rec + 1], dx=DT)
        time_lost = max(sg.t[i_rec] - sg.t[a] - dist / v_pre, 0.05)
        rec_t = max(sg.t[i_rec] - sg.t[i_min], DT)
        out.append(Cut(turn=float(ang[i]), v_pre=float(v_pre), v_min=float(sg.v[i_min]),
                       elapsed=float(sg.t[i_rec] - sg.t[a]), dist=float(dist), time_lost=float(time_lost),
                       cod_eff=float(abs(ang[i]) / time_lost),
                       recovery_acc=float((sg.v[i_rec] - sg.v[i_min]) / rec_t)))
    return out


def attempt_features(x, y, s):
    sg = signals(x, y, s)
    if sg is None:
        return None
    f = AttemptFeatures()
    f.peak_speed, f.accel_eff = acceleration_efficiency(sg)
    f.decel_ctrl = deceleration_control(sg)
    f.smoothness = smoothness(sg)
    f.cuts = cuts(sg)
    return f


In [ ]:
%%writefile src/features/combine.py
"""Build player-level Combine movement features and the MEI.

Usage: python -P -m src.features.combine
Writes data/processed/combine_attempts.parquet, combine_cuts.parquet, combine_features.parquet.
"""

import numpy as np
import pandas as pd
from scipy.stats import norm

from src.features.kinematics import attempt_features

from src.paths import PROC as OUT, RAW

COMPONENTS = ["accel_eff", "decel_ctrl", "cod_eff", "smoothness", "recovery_acc"]
MEI_WEIGHTS = {"accel_eff": 0.25, "decel_ctrl": 0.20, "cod_eff": 0.25,
               "smoothness": 0.15, "recovery_acc": 0.15}
MIN_PLAYERS_PER_DRILL = 10
# three-cone cuts mix reversals and rounded weaves on 5-yard legs; per-cut COD there
# runs opposite to the official time even after controlling for speed (docs/feature_notes.md)
NO_CUT_SCORING = {"THREE_CONE_DRILL"}

# the same drill appears under more than one name across Combine years
DRILL_ALIASES = {
    "PASS_PRO-MIRROR_DRILL": "PASS_PRO_MIRROR_DRILL",
    "LINE": "LINE_DRILL",
    "SPEED_OUT_LEFT": "SPEED_OUT_ROUTE_LEFT",
    "OVER_THE_SHOULDER_ADJUST": "OVER_SHOULDER_ADJUST",
}


def load_tracking():
    t = pd.read_csv(RAW / "combine_tracking.csv", parse_dates=["time"])
    t = t[(t.entity_type == "PLAYER") & (t.drill_type != "SKILL_DRILLS_LB")]
    t["drill"] = t.drill_name.replace(DRILL_ALIASES)
    return t.sort_values(["nfl_id", "drill", "attempt", "time"])


def attempt_tables(t):
    attempts, cuts = [], []
    for (pid, dtype, drill, att), g in t.groupby(["nfl_id", "drill_type", "drill", "attempt"], sort=False):
        f = attempt_features(g.x.to_numpy(), g.y.to_numpy(), g.s.to_numpy())
        if f is None:
            continue
        key = dict(nfl_id=pid, drill_type=dtype, drill=drill, attempt=att)
        attempts.append({**key, "peak_speed": f.peak_speed, "accel_eff": f.accel_eff,
                         "decel_ctrl": f.decel_ctrl, "smoothness": f.smoothness, "n_cuts": len(f.cuts),
                         "cod_eff": np.median([c.cod_eff for c in f.cuts]) if f.cuts else np.nan,
                         "recovery_acc": np.median([c.recovery_acc for c in f.cuts]) if f.cuts else np.nan})
        cuts += [{**key, **vars(c)} for c in f.cuts]
    return pd.DataFrame(attempts), pd.DataFrame(cuts)


def normal_scores(s):
    """Rank-based z-scores: robust to the heavy tails of ratio metrics."""
    r = s.rank(pct=True, method="average")
    n = s.notna().sum()
    return pd.Series(norm.ppf((r * n - 0.5) / n), index=s.index)


def drill_scaled(df, cols, by="drill"):
    """Normal-score each column within drill, dropping drills with too few players."""
    keep = df.groupby(by).nfl_id.transform("nunique") >= MIN_PLAYERS_PER_DRILL
    df = df[keep].copy()
    for c in cols:
        df[c] = df.groupby(by)[c].transform(normal_scores)
    return df


def player_features(attempts, cuts, results):
    per_drill = attempts.groupby(["nfl_id", "drill_type", "drill"], as_index=False)[COMPONENTS + ["peak_speed"]].mean()
    per_drill.loc[per_drill.drill.isin(NO_CUT_SCORING), ["cod_eff", "recovery_acc"]] = np.nan
    z = drill_scaled(per_drill, COMPONENTS)
    feats = z.groupby("nfl_id")[COMPONENTS].mean()
    feats["n_drills"] = z.groupby("nfl_id").drill.nunique()

    # 40-yard dash: top speed and start quality from the best attempt
    forty = attempts[attempts.drill == "FORTY_YARD_DASH"].groupby("nfl_id").agg(
        top_speed=("peak_speed", "max"), forty_accel_eff=("accel_eff", "max"))
    feats = feats.join(forty)

    # left/right symmetry of change-of-direction efficiency
    # (reversals near 180 deg have no meaningful side, so only 35-150 deg cuts count)
    sided = cuts[cuts.turn.abs().between(35, 150)]
    c = drill_scaled(sided.assign(side=np.where(sided.turn > 0, "L", "R")), ["cod_eff"])
    side = c.pivot_table(index="nfl_id", columns="side", values="cod_eff", aggfunc="mean")
    n_side = c.pivot_table(index="nfl_id", columns="side", values="cod_eff", aggfunc="size")
    ok = (n_side.min(axis=1) >= 3)
    feats["cod_asymmetry"] = (side.L - side.R).abs().where(ok)

    feats = results[["nfl_id", "draft_year", "combine_position"]].merge(feats.reset_index(), on="nfl_id", how="left")
    # MEI: standardise components within position group, weight, re-standardise
    zpos = feats.groupby("combine_position")[COMPONENTS].transform(lambda s: (s - s.mean()) / s.std())
    mei = sum(w * zpos[c].fillna(0) for c, w in MEI_WEIGHTS.items())
    mei[zpos.isna().all(axis=1)] = np.nan
    feats["mei"] = mei.groupby(feats.combine_position).transform(lambda s: (s - s.mean()) / s.std())
    return feats


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    results = pd.read_csv(RAW / "combine_results.csv")
    attempts, cuts = attempt_tables(load_tracking())
    attempts.to_parquet(OUT / "combine_attempts.parquet", index=False)
    cuts.to_parquet(OUT / "combine_cuts.parquet", index=False)
    feats = player_features(attempts, cuts, results)
    feats.to_parquet(OUT / "combine_features.parquet", index=False)
    print(f"attempts={len(attempts)} cuts={len(cuts)} players={feats.mei.notna().sum()}/{len(feats)}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/outcomes.py
"""Build player-level NFL regular-season outcomes.

Usage: PYTHONPATH=$PWD python -P -m src.outcomes [--skip-tracking]
Writes data/processed/db_coverage_plays.parquet and data/processed/outcomes.parquet.
"""
import sys

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter

from src.paths import PROC as OUT, RAW
SEASONS = (2023, 2024, 2025)
LAST_SEASON = 2025
GAMES_PER_SEASON = 17
DT = 0.1
DB_WINDOW = 15          # frames after pass_forward used for DB reaction metrics


def load_plays():
    games = pd.read_csv(RAW / "games.csv")
    reg = games.loc[games.season_type == "REG", ["game_id", "season"]]
    pp = pd.read_csv(RAW / "player_play.csv", low_memory=False).merge(reg, on="game_id")
    res = pd.read_csv(RAW / "combine_results.csv")[["nfl_id", "combine_position"]]
    return pp.merge(res, on="nfl_id")


def shrink(df, value, n, sd=None, rate=False):
    """Empirical-Bayes shrinkage of per-player means toward the position-group mean.

    `value` is the raw per-player mean over `n` plays; `sd` is the per-play standard
    deviation column (ignored for rates, which use p(1 - p)).
    """
    out = pd.Series(np.nan, index=df.index)
    for _, g in df.groupby("combine_position"):
        g = g[g[n] > 0]
        if len(g) < 5:
            continue
        mu = np.average(g[value], weights=g[n])
        if rate:
            noise = (mu * (1 - mu)) / g[n]
        else:
            noise = (np.average(g[sd].fillna(g[sd].median()) ** 2, weights=g[n])) / g[n]
        tau2 = max(np.var(g[value]) - noise.mean(), 1e-9)
        w = tau2 / (tau2 + noise)
        out[g.index] = w * g[value] + (1 - w) * mu
    return out


def per_player(rows, value, name, rate=False):
    """Aggregate a per-play value to mean, n, sd and its shrunk estimate."""
    agg = rows.groupby(["nfl_id", "combine_position"])[value].agg(["mean", "size", "std"]).reset_index()
    agg.columns = ["nfl_id", "combine_position", name, f"n_{name}", "sd"]
    agg[f"{name}_eb"] = shrink(agg, name, f"n_{name}", "sd", rate=rate)
    return agg.drop(columns="sd")


def receiving(pp):
    routes = pp[pp.combine_position.isin(["WR", "TE"]) & pp.route_ran.notna()].copy()
    routes["rec_yards"] = routes.rec_yards.where(routes.target, 0).fillna(0)
    sep = per_player(routes.dropna(subset=["separation_at_pass_forward"]), "separation_at_pass_forward", "separation")
    yprr = per_player(routes, "rec_yards", "yprr")
    routes["epa_route"] = routes.expected_points_added.where(routes.target, 0)
    epa = per_player(routes, "epa_route", "epa_per_route")
    return [sep, yprr, epa]


def pass_block(pp):
    pb = pp[pp.combine_position.isin(["OL", "TE"]) & pp.pressure_allowed.notna()].copy()
    pb["pressure_allowed"] = pb.pressure_allowed.astype(float)
    pb["sack_allowed"] = pb.sack_allowed.astype(float)
    return [per_player(pb, "pressure_allowed", "pressure_allowed_rate", rate=True),
            per_player(pb[pb.combine_position == "OL"], "sack_allowed", "sack_allowed_rate", rate=True),
            per_player(pb[pb.combine_position == "OL"].dropna(subset=["peak_pressure_probability_allowed"]),
                       "peak_pressure_probability_allowed", "peak_pressure_prob")]


def pass_rush(pp):
    pr = pp[(pp.combine_position == "DL") & pp.sack.notna()].copy()
    pr["sack"] = pr.sack.astype(float)
    pr["pressure"] = (pr.time_to_pressure.notna() | (pr.sack > 0)).astype(float)
    get_off = pp[(pp.combine_position == "DL") & pp.player_get_off.notna()]
    return [per_player(pr, "pressure", "pressure_rate", rate=True),
            per_player(pr, "sack", "sack_rate", rate=True),
            per_player(get_off, "player_get_off", "get_off")]


def db_cushion(pp):
    db = pp[(pp.combine_position == "DB") & pp.cushion.notna()]
    return [per_player(db, "cushion", "cushion")]


def db_coverage_plays(pp):
    """Per-play DB reaction after the throw in man coverage, from game tracking."""
    man = pp[(pp.combine_position == "DB") & (pp.coverage_assignment == "MAN") & pp.pass_result.isin(["C", "I", "IN"])]
    keys = man[["game_id", "play_id", "nfl_id"]].drop_duplicates()
    want = set(map(tuple, keys.to_numpy()))
    ids = set(keys.nfl_id)
    out = []
    for season in SEASONS:
        reader = pd.read_csv(RAW / f"game_tracking_{season}.csv", usecols=["game_id", "play_id", "nfl_id", "time", "s", "event"],
                             chunksize=2_000_000)
        frames = []
        for chunk in reader:
            chunk = chunk[chunk.nfl_id.isin(ids)]
            k = list(zip(chunk.game_id, chunk.play_id, chunk.nfl_id))
            frames.append(chunk[[x in want for x in k]])
        f = pd.concat(frames).sort_values(["game_id", "play_id", "nfl_id", "time"])
        for (gid, pid, nid), g in f.groupby(["game_id", "play_id", "nfl_id"], sort=False):
            ev = g.event.to_numpy()
            throw = np.flatnonzero(ev == "pass_forward")
            if not len(throw) or throw[0] + DB_WINDOW > len(g):
                continue
            s = g.s.to_numpy()
            i0 = throw[0]
            seg = s[max(i0 - 3, 0):i0 + DB_WINDOW]
            acc = savgol_filter(seg, 5, 2, deriv=1, delta=DT)[min(i0, 3):]
            out.append(dict(game_id=gid, play_id=pid, nfl_id=nid, season=season,
                            break_accel=float(acc.max()),
                            closing_speed=float(s[i0:i0 + DB_WINDOW].max()),
                            speed_at_throw=float(s[i0])))
        print(f"season {season}: {len(out)} DB man-coverage plays so far", flush=True)
    return pd.DataFrame(out)


def db_tracking(cov):
    cov = cov.assign(combine_position="DB")
    return [per_player(cov, "break_accel", "db_break_accel"),
            per_player(cov, "closing_speed", "db_closing_speed")]


def career():
    c = pd.read_csv(RAW / "player_career_successes.csv")
    p = pd.read_csv(RAW / "players.csv")[["nfl_id", "draft_year", "draft_overall_pick"]]
    c = c.merge(p, on="nfl_id")
    seasons = LAST_SEASON - c.draft_year + 1
    c["snaps_per_season"] = (c.career_offensive_snaps + c.career_defensive_snaps) / seasons
    c["games_active_share"] = (c.career_games_active / (seasons * GAMES_PER_SEASON)).clip(upper=1)
    c["starts_per_season"] = c.career_games_started / seasons
    c["honors"] = c.ap_all_pro_1st_team + c.ap_all_pro_2nd_team + c.pro_bowl_original_ballot
    return c[["nfl_id", "draft_overall_pick", "snaps_per_season", "games_active_share", "starts_per_season", "honors"]]


def main():
    pp = load_plays()
    tables = receiving(pp) + pass_block(pp) + pass_rush(pp) + db_cushion(pp)
    cov_path = OUT / "db_coverage_plays.parquet"
    if "--skip-tracking" not in sys.argv or not cov_path.exists():
        db_coverage_plays(pp).to_parquet(cov_path, index=False)
    tables += db_tracking(pd.read_parquet(cov_path))

    res = pd.read_csv(RAW / "combine_results.csv")[["nfl_id", "draft_year", "combine_position"]]
    out = res.merge(career(), on="nfl_id", how="left")
    for t in tables:
        out = out.merge(t.drop(columns="combine_position"), on="nfl_id", how="left")
    out.to_parquet(OUT / "outcomes.parquet", index=False)
    print(out.groupby("combine_position").count().T.to_string())


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/models.py
"""Phase 4: does Combine tracking predict NFL outcomes beyond the stopwatch and draft slot?

Usage: PYTHONPATH=$PWD python -P -m src.models [--perms N]
Writes data/processed/model_results.csv and data/processed/component_assoc.csv.
"""
import sys
import warnings

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.impute import SimpleImputer
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.features.combine import COMPONENTS

from src.paths import PROC, RAW
SEED = 2027
UNDRAFTED_PICK = 262
N_BOOT = 1000

TRADITIONAL = ["forty", "ten_yd_split", "vertical", "broad_jump", "combine_weight", "combine_height"]
TRACKING = COMPONENTS                  # learned weights
MEI = ["mei"]                          # writeup's fixed weights
DRAFT = ["log_pick", "undrafted"]

# (position, outcome, min plays, plays column); outcomes chosen in docs/outcome_notes.md
TASKS = [
    ("WR", "separation_eb", 50, "n_separation"), ("WR", "yprr_eb", 50, "n_yprr"),
    ("TE", "separation_eb", 50, "n_separation"), ("TE", "yprr_eb", 50, "n_yprr"),
    ("OL", "peak_pressure_prob_eb", 100, "n_peak_pressure_prob"),
    ("OL", "pressure_allowed_rate_eb", 100, "n_pressure_allowed_rate"),
    ("DL", "get_off_eb", 50, "n_get_off"), ("DL", "pressure_rate_eb", 50, "n_pressure_rate"),
    ("DL", "sack_rate_eb", 50, "n_sack_rate"),
    ("DB", "db_closing_speed_eb", 15, "n_db_closing_speed"), ("DB", "db_break_accel_eb", 15, "n_db_break_accel"),
] + [(p, o, 0, None) for p in ["WR", "TE", "OL", "DL", "DB"] for o in ["snaps_per_season", "games_active_share"]]

MODELS = {
    "M0 draft only": DRAFT,
    "M1 traditional": TRADITIONAL,
    "M2 tracking": TRACKING,
    "M2b MEI (fixed weights)": MEI,
    "M3 hybrid": TRADITIONAL + TRACKING,
    "M1 + draft": TRADITIONAL + DRAFT,
    "M2 + draft": TRACKING + DRAFT,
    "M3 + draft": TRADITIONAL + TRACKING + DRAFT,
}


def load():
    feats = pd.read_parquet(PROC / "combine_features.parquet")
    outc = pd.read_parquet(PROC / "outcomes.parquet").drop(columns=["draft_year", "combine_position"])
    trad = pd.read_csv(RAW / "combine_results.csv")[["nfl_id"] + TRADITIONAL]
    df = feats.merge(trad, on="nfl_id").merge(outc, on="nfl_id")
    df["undrafted"] = df.draft_overall_pick.isna().astype(float)
    df["log_pick"] = np.log(df.draft_overall_pick.fillna(UNDRAFTED_PICK))
    return df


def model():
    return make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                         RidgeCV(alphas=np.logspace(-2, 4, 25)))


def cv_predict(X, y, groups):
    """Leave-one-draft-class-out out-of-sample predictions."""
    pred = np.full(len(y), np.nan)
    for g in np.unique(groups):
        tr, te = groups != g, groups == g
        pred[te] = model().fit(X[tr], y[tr]).predict(X[te])
    return pred


def boot_ci(y, preds_a, preds_b=None, rng=None):
    """Bootstrap 95% CI of OOS R^2 (or of the R^2 difference a - b)."""
    n, stats = len(y), []
    for _ in range(N_BOOT):
        i = rng.integers(0, n, n)
        r = r2_score(y[i], preds_a[i])
        if preds_b is not None:
            r -= r2_score(y[i], preds_b[i])
        stats.append(r)
    return np.percentile(stats, [2.5, 97.5])


def perm_p(X, y, groups, observed, n_perm, rng):
    """Permutation p-value: shuffle the outcome within draft class, re-run CV."""
    if n_perm == 0:
        return np.nan
    hits = 0
    for _ in range(n_perm):
        yp = y.copy()
        for g in np.unique(groups):
            m = groups == g
            yp[m] = rng.permutation(yp[m])
        hits += r2_score(yp, cv_predict(X, yp, groups)) >= observed
    return (hits + 1) / (n_perm + 1)


def run(n_perm):
    df = load()
    rng = np.random.default_rng(SEED)
    rows, assoc = [], []
    for pos, outcome, min_n, ncol in TASKS:
        d = df[(df.combine_position == pos) & df[outcome].notna()]
        if ncol:
            d = d[d[ncol] >= min_n]
        y, groups = d[outcome].to_numpy(float), d.draft_year.to_numpy()
        preds = {}
        for name, cols in MODELS.items():
            X = d[cols].to_numpy(float)
            preds[name] = p = cv_predict(X, y, groups)
            r2 = r2_score(y, p)
            lo, hi = boot_ci(y, p, rng=rng)
            test_perm = name in ("M2 tracking", "M2b MEI (fixed weights)", "M1 traditional")
            rows.append(dict(position=pos, outcome=outcome, model=name, n=len(y), r2=r2, r2_lo=lo, r2_hi=hi,
                             mae=mean_absolute_error(y, p), rho=spearmanr(y, p).statistic,
                             perm_p=perm_p(X, y, groups, r2, n_perm if test_perm else 0, rng)))
        for a, b in [("M3 hybrid", "M1 traditional"), ("M3 + draft", "M1 + draft"), ("M2 + draft", "M0 draft only")]:
            lo, hi = boot_ci(y, preds[a], preds[b], rng=rng)
            rows.append(dict(position=pos, outcome=outcome, model=f"Δ {a} − {b}", n=len(y),
                             r2=r2_score(y, preds[a]) - r2_score(y, preds[b]), r2_lo=lo, r2_hi=hi))
        # single-feature associations, raw and holding draft slot fixed
        for f in TRACKING + ["mei", "cod_asymmetry", "top_speed"] + TRADITIONAL:
            x = d[f]
            ok = x.notna()
            if ok.sum() < 15:
                continue
            rho, p = spearmanr(x[ok], y[ok])
            resid = lambda v: v - np.polyval(np.polyfit(d.log_pick[ok], v, 1), d.log_pick[ok])
            prho, pp = spearmanr(resid(x[ok].rank()), resid(pd.Series(y[ok.to_numpy()]).rank().to_numpy()))
            assoc.append(dict(position=pos, outcome=outcome, feature=f, n=int(ok.sum()),
                              rho=rho, p=p, partial_rho_pick=prho, partial_p=pp))
        print(f"{pos} {outcome}: n={len(y)}", flush=True)
    return pd.DataFrame(rows), pd.DataFrame(assoc)


def main():
    warnings.filterwarnings("ignore", category=RuntimeWarning)
    n_perm = int(sys.argv[sys.argv.index("--perms") + 1]) if "--perms" in sys.argv else 200
    res, assoc = run(n_perm)
    res.to_csv(PROC / "model_results.csv", index=False)
    assoc.to_csv(PROC / "component_assoc.csv", index=False)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/analysis.py
"""Phase 5: learned MEI v2, transfer test, bootstrap CIs and coverage for the writeup.

Usage: PYTHONPATH=$PWD python -P -m src.analysis
Writes data/processed/analysis/*.csv.
"""
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

from src import models as M

OUT = M.PROC / "analysis"
N_BOOT = 2000

# primary outcome per position; "sign" orients so higher = better
PRIMARY = {"DL": ("get_off_eb", -1), "DB": ("db_closing_speed_eb", 1), "WR": ("separation_eb", 1),
           "TE": ("separation_eb", 1), "OL": ("peak_pressure_prob_eb", -1)}
COMPARE = [("DL", "get_off_eb"), ("DL", "pressure_rate_eb"), ("DL", "sack_rate_eb"),
           ("DB", "db_closing_speed_eb"), ("WR", "separation_eb"), ("OL", "peak_pressure_prob_eb")]
COMPARE_MODELS = ["M0 draft only", "M1 traditional", "M2 tracking", "M2b MEI (fixed weights)", "M3 hybrid"]


def task_frame(df, pos, outcome):
    spec = next(t for t in M.TASKS if t[0] == pos and t[1] == outcome)
    d = df[(df.combine_position == pos) & df[outcome].notna()]
    return d[d[spec[3]] >= spec[2]] if spec[3] else d


def boot_rho(y, p, rng):
    n = len(y)
    stats = [spearmanr(y[i], p[i]).statistic for i in (rng.integers(0, n, n) for _ in range(N_BOOT))]
    return np.percentile(stats, [2.5, 97.5])


def model_comparison(df, rng):
    rows = []
    for pos, outcome in COMPARE:
        d = task_frame(df, pos, outcome)
        y, g = d[outcome].to_numpy(float), d.draft_year.to_numpy()
        for name in COMPARE_MODELS:
            p = M.cv_predict(d[M.MODELS[name]].to_numpy(float), y, g)
            lo, hi = boot_rho(y, p, rng)
            rows.append(dict(position=pos, outcome=outcome, model=name, n=len(y),
                             rho=spearmanr(y, p).statistic, lo=lo, hi=hi))
    return pd.DataFrame(rows)


def learned_weights(df):
    """MEI v2 weights: ridge on the five components, per position, primary outcome, full data."""
    rows = []
    for pos, (outcome, sign) in PRIMARY.items():
        d = task_frame(df, pos, outcome)
        m = M.model().fit(d[M.TRACKING].to_numpy(float), d[outcome].to_numpy(float))
        coef = sign * m[-1].coef_
        rows += [dict(position=pos, outcome=outcome, component=c, weight=w / np.abs(coef).sum())
                 for c, w in zip(M.TRACKING, coef)]
    return pd.DataFrame(rows)


def transfer(df, rng):
    """Train the DL index on get-off in two draft classes; score the third on pass-rush outcomes."""
    dl = df[df.combine_position == "DL"].copy()
    train_rows = task_frame(df, "DL", "get_off_eb").nfl_id
    dl["index"] = np.nan
    for year in dl.draft_year.unique():
        tr = dl[dl.nfl_id.isin(train_rows) & (dl.draft_year != year)]
        m = M.model().fit(tr[M.TRACKING].to_numpy(float), -tr.get_off_eb.to_numpy(float))
        te = dl.draft_year == year
        dl.loc[te, "index"] = m.predict(dl.loc[te, M.TRACKING].to_numpy(float))
    rows = []
    for outcome in ["get_off_eb", "pressure_rate_eb", "sack_rate_eb", "snaps_per_season"]:
        d = task_frame(dl.assign(combine_position="DL"), "DL", outcome)
        y, x = d[outcome].to_numpy(float), d["index"].to_numpy(float)
        lo, hi = boot_rho(y, x, rng)
        resid = lambda v: v - np.polyval(np.polyfit(d.log_pick, v, 1), d.log_pick)
        rows.append(dict(outcome=outcome, n=len(y), rho=spearmanr(x, y).statistic, lo=lo, hi=hi,
                         partial_rho_pick=spearmanr(resid(pd.Series(x).rank().to_numpy()),
                                                    resid(pd.Series(y).rank().to_numpy())).statistic))
    dl[["nfl_id", "draft_year", "index", "accel_eff", "smoothness", "get_off_eb", "pressure_rate_eb",
        "sack_rate_eb", "n_get_off", "n_pressure_rate", "forty", "draft_overall_pick"]].to_csv(OUT / "dl_players.csv", index=False)
    return pd.DataFrame(rows)


def coverage(df):
    cols = {"Official 40-yard dash": "forty", "Official 10-yd split": "ten_yd_split", "Official vertical": "vertical",
            "Official three-cone": "three_cone", "Official short shuttle": "short_shuttle",
            "Tracking: acceleration": "accel_eff", "Tracking: change of direction": "cod_eff"}
    trad = pd.read_csv(M.RAW / "combine_results.csv")[["nfl_id", "three_cone", "short_shuttle"]]
    d = df.merge(trad, on="nfl_id")
    return pd.DataFrame([dict(metric=k, share=d[v].notna().mean(), n=int(d[v].notna().sum())) for k, v in cols.items()])


def validation():
    """Tracked drill metrics vs. the official stopwatch (Spearman)."""
    a = pd.read_parquet(M.PROC / "combine_attempts.parquet")
    res = pd.read_csv(M.RAW / "combine_results.csv").set_index("nfl_id")
    best = lambda drill, col, how: getattr(a[a.drill == drill].groupby("nfl_id")[col], how)()
    checks = [("Tracked 40 top speed vs. official 40", best("FORTY_YARD_DASH", "peak_speed", "max"), "forty"),
              ("40 acceleration efficiency vs. 10-yard split", best("FORTY_YARD_DASH", "accel_eff", "max"), "ten_yd_split"),
              ("Three-cone acceleration efficiency vs. three-cone", best("THREE_CONE_DRILL", "accel_eff", "mean"), "three_cone"),
              ("Shuttle COD efficiency vs. shuttle", best("SHORT_SHUTTLE", "cod_eff", "mean"), "short_shuttle")]
    rows = []
    for name, x, col in checks:
        d = pd.concat([x.rename("x"), res[col]], axis=1).dropna()
        rows.append(dict(check=name, n=len(d), rho=spearmanr(d.x, d[col]).statistic))
    return pd.DataFrame(rows)


def reliability():
    """Split-half (odd/even game) reliability of each outcome, Spearman-Brown corrected."""
    from src import outcomes as O
    pp = O.load_plays()
    routes = pp[pp.combine_position.isin(["WR", "TE"]) & pp.route_ran.notna()].copy()
    routes["rec_yards"] = routes.rec_yards.where(routes.target, 0).fillna(0)
    routes["epa_route"] = routes.expected_points_added.where(routes.target, 0)
    pb = pp[(pp.combine_position == "OL") & pp.pressure_allowed.notna()].astype({"pressure_allowed": float, "sack_allowed": float})
    pr = pp[(pp.combine_position == "DL") & pp.sack.notna()].copy()
    pr["sack"] = pr.sack.astype(float)
    pr["pressure"] = (pr.time_to_pressure.notna() | (pr.sack > 0)).astype(float)
    cov = pd.read_parquet(M.PROC / "db_coverage_plays.parquet")
    specs = [("WR/TE separation", routes.dropna(subset=["separation_at_pass_forward"]), "separation_at_pass_forward"),
             ("WR/TE yards per route", routes, "rec_yards"), ("WR/TE EPA per route", routes, "epa_route"),
             ("OL pressure allowed", pb, "pressure_allowed"), ("OL sacks allowed", pb, "sack_allowed"),
             ("OL peak pressure prob.", pb.dropna(subset=["peak_pressure_probability_allowed"]), "peak_pressure_probability_allowed"),
             ("DL pressure rate", pr, "pressure"), ("DL sack rate", pr, "sack"),
             ("DL get-off", pp[(pp.combine_position == "DL") & pp.player_get_off.notna()], "player_get_off"),
             ("DB cushion", pp[(pp.combine_position == "DB") & pp.cushion.notna()], "cushion"),
             ("DB break accel", cov, "break_accel"), ("DB closing speed", cov, "closing_speed")]
    rows = []
    for name, d, v in specs:
        half = d.game_id.rank(method="dense") % 2
        h = d.groupby(["nfl_id", half])[v].agg(["mean", "size"]).unstack()
        h = h[(h["size"] >= 20).all(axis=1)]
        r = np.corrcoef(h["mean"][0], h["mean"][1])[0, 1]
        rows.append(dict(outcome=name, players=len(h), split_half_r=r, reliability=2 * r / (1 + r)))
    return pd.DataFrame(rows)


def main():
    OUT.mkdir(exist_ok=True)
    rng = np.random.default_rng(M.SEED)
    df = M.load()
    for name, table in [("validation", validation()), ("reliability", reliability()),
                        ("model_comparison", model_comparison(df, rng)), ("learned_weights", learned_weights(df)),
                        ("transfer", transfer(df, rng)), ("coverage", coverage(df))]:
        table.to_csv(OUT / f"{name}.csv", index=False)
        print(f"\n== {name}\n{table.round(3).to_string(index=False)}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/figures.py
"""Writeup figures from Phase 4/5 outputs.

Usage: PYTHONPATH=$PWD python -P -m src.figures
Writes figures/*.png.
"""
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src import models as M
from src.features.combine import MEI_WEIGHTS

from src.paths import FIG

A = M.PROC / "analysis"

SURFACE, INK, INK2, GRID = "#0c2a1f", "#f0efec", "#c3c2b7", "#2b4a3e"
# dark-mode categorical slots 1-4 (validated on SURFACE)
BLUE, ORANGE, AQUA, YELLOW = "#3987e5", "#d95926", "#199e70", "#c98500"
MODEL_STYLE = {"M1 traditional": ("Traditional (stopwatch)", BLUE, "o"),
               "M2 tracking": ("Tracking (learned weights)", ORANGE, "D"),
               "M3 hybrid": ("Hybrid", AQUA, "s"),
               "M2b MEI (fixed weights)": ("MEI, writeup weights", YELLOW, "^")}
OUTCOME_LABEL = {("DL", "get_off_eb"): "DL get-off", ("DL", "pressure_rate_eb"): "DL pressure rate",
                 ("DL", "sack_rate_eb"): "DL sack rate", ("DB", "db_closing_speed_eb"): "DB closing speed",
                 ("WR", "separation_eb"): "WR separation", ("OL", "peak_pressure_prob_eb"): "OL pressure prob."}

plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "text.color": INK, "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "axes.edgecolor": GRID, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False, "font.family": "DejaVu Sans",
    "font.size": 11, "axes.titlesize": 14, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "legend.frameon": False, "lines.linewidth": 2,
})


def title(fig, main, sub):
    fig.text(0.02, 0.965, main, fontsize=16, fontweight="bold", color=INK, va="top")
    fig.text(0.02, 0.905, sub, fontsize=11, color=INK2, va="top")


def save(fig, name):
    FIG.mkdir(exist_ok=True)
    fig.savefig(FIG / name, dpi=200)
    plt.close(fig)
    print("saved", FIG / name)


def trend(ax, x, y, color):
    ok = np.isfinite(x) & np.isfinite(y)
    b = np.polyfit(x[ok], y[ok], 1)
    xs = np.linspace(x[ok].min(), x[ok].max(), 50)
    ax.plot(xs, np.polyval(b, xs), color=INK2, lw=2, ls="--", zorder=2)


def scatter(ax, x, y, color):
    ax.scatter(x, y, s=46, color=color, edgecolor=SURFACE, linewidth=1.5, zorder=3)
    trend(ax, np.asarray(x, float), np.asarray(y, float), color)


def fig_coverage():
    c = pd.read_csv(A / "coverage.csv").iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 4.6))
    fig.subplots_adjust(left=0.33, right=0.93, top=0.8, bottom=0.12)
    colors = [ORANGE if m.startswith("Tracking") else BLUE for m in c.metric]
    ax.barh(c.metric, c.share * 100, color=colors, height=0.62, edgecolor=SURFACE, linewidth=2)
    for i, (s, n) in enumerate(zip(c.share, c.n)):
        ax.text(s * 100 + 1.5, i, f"{s:.0%}  (n={n})", va="center", color=INK, fontsize=10)
    ax.set_xlim(0, 118)
    ax.set_xlabel("Share of the 510 Combine players with the metric (%)")
    ax.grid(axis="y", visible=False)
    title(fig, "Tracking measures every player", "Nearly two in three prospects skip the official three-cone or shuttle; tracked drills cover them all.")
    save(fig, "fig1_coverage.png")


def fig_transfer():
    d = pd.read_csv(A / "dl_players.csv")
    d["index"] = (d["index"] - d["index"].mean()) / d["index"].std()
    t = pd.read_csv(A / "transfer.csv").set_index("outcome")
    fig, axes = plt.subplots(1, 2, figsize=(10, 4.8))
    fig.subplots_adjust(left=0.08, right=0.98, top=0.72, bottom=0.14, wspace=0.28)
    g = d[d.n_get_off >= 50]
    scatter(axes[0], g["index"], g.get_off_eb, ORANGE)
    axes[0].invert_yaxis()
    axes[0].set_ylabel("NFL get-off time (s, quicker ↑)")
    r = t.loc["get_off_eb"]
    axes[0].set_title(f"Trained on: get-off\nρ = {-r.rho:.2f}  [95% CI {-r.hi:.2f}, {-r.lo:.2f}]", fontsize=12)
    p = d[d.n_pressure_rate >= 50]
    scatter(axes[1], p["index"], p.pressure_rate_eb * 100, ORANGE)
    axes[1].set_ylabel("NFL pressure rate (%, shrunk estimate)")
    r = t.loc["pressure_rate_eb"]
    axes[1].set_title(f"Never trained on: pressure rate\nρ = {r.rho:.2f}  [95% CI {r.lo:.2f}, {r.hi:.2f}]", fontsize=12)
    for ax in axes:
        ax.set_xlabel("DL Explosiveness Index (out-of-sample, z-score)")
    title(fig, "Combine explosiveness transfers to the pass rush",
          "Index learned from Combine tracking on two draft classes, scored on the third (n = 82 / 79 DL).")
    save(fig, "fig2_transfer.png")


def fig_smoothness():
    d = pd.read_csv(A / "dl_players.csv")
    a = pd.read_csv(M.PROC / "component_assoc.csv")
    r = a[(a.position == "DL") & (a.outcome == "pressure_rate_eb") & (a.feature == "smoothness")].iloc[0]
    p = d[d.n_pressure_rate >= 50]
    fig, ax = plt.subplots(figsize=(9, 4.8))
    fig.subplots_adjust(left=0.1, right=0.97, top=0.8, bottom=0.14)
    scatter(ax, p.smoothness, p.pressure_rate_eb * 100, YELLOW)
    ax.set_xlabel("Combine movement smoothness (drill-standardised; smoother →)")
    ax.set_ylabel("NFL pressure rate (%, shrunk estimate)")
    ax.text(0.98, 0.94, f"ρ = {r.rho:.2f}\nρ given draft slot = {r.partial_rho_pick:.2f}\nn = {int(r.n)}",
            transform=ax.transAxes, ha="right", va="top", color=INK, fontsize=11)
    title(fig, "The smoothness paradox", "Smoother Combine movers generate fewer NFL pressures. Explosive movement is variable movement.")
    save(fig, "fig3_smoothness.png")


def fig_models():
    c = pd.read_csv(A / "model_comparison.csv")
    tasks = list(OUTCOME_LABEL)
    fig, ax = plt.subplots(figsize=(9, 5.4))
    fig.subplots_adjust(left=0.2, right=0.97, top=0.74, bottom=0.12)
    offsets = dict(zip(MODEL_STYLE, [0.27, 0.09, -0.09, -0.27]))
    for model, (label, color, marker) in MODEL_STYLE.items():
        for i, (pos, out) in enumerate(tasks):
            r = c[(c.position == pos) & (c.outcome == out) & (c.model == model)].iloc[0]
            y = len(tasks) - 1 - i + offsets[model]
            ax.plot([r.lo, r.hi], [y, y], color=color, lw=2, solid_capstyle="round")
            ax.scatter(r.rho, y, s=60, color=color, marker=marker, edgecolor=SURFACE, linewidth=1.5, zorder=3,
                       label=label if i == 0 else None)
    ax.axvline(0, color=INK2, lw=1)
    ax.set_yticks(range(len(tasks)))
    ax.set_yticklabels([OUTCOME_LABEL[t] for t in tasks][::-1])
    ax.grid(axis="y", visible=False)
    ax.set_xlabel("Out-of-sample Spearman ρ, predicted vs. actual (95% bootstrap CI)")
    ax.legend(loc="lower center", bbox_to_anchor=(0.45, 1.0), ncol=4, fontsize=9.5, handletextpad=0.3, columnspacing=1.2)
    title(fig, "Where Combine movement predicts NFL performance",
          "Leave-one-draft-class-out ridge models. Strong for defensive line, weak or absent elsewhere.")
    save(fig, "fig4_models.png")


def fig_weights():
    w = pd.read_csv(A / "learned_weights.csv")
    w = w[w.position == "DL"].set_index("component").weight
    prior = pd.Series(MEI_WEIGHTS)
    names = {"accel_eff": "Acceleration", "decel_ctrl": "Deceleration", "cod_eff": "Change of direction",
             "smoothness": "Smoothness", "recovery_acc": "Recovery"}
    order = list(names)[::-1]
    y = np.arange(len(order))
    fig, ax = plt.subplots(figsize=(8, 4.8))
    fig.subplots_adjust(left=0.24, right=0.95, top=0.74, bottom=0.13)
    ax.barh(y + 0.19, prior[order], height=0.36, color=BLUE, edgecolor=SURFACE, linewidth=2, label="Writeup (a priori)")
    ax.barh(y - 0.19, w[order], height=0.36, color=ORANGE, edgecolor=SURFACE, linewidth=2, label="Learned for DL (get-off)")
    for yy, v in zip(y - 0.19, w[order]):
        ax.text(v + (0.012 if v >= 0 else -0.012), yy, f"{v:+.2f}", va="center", ha="left" if v >= 0 else "right", fontsize=9.5, color=INK)
    ax.axvline(0, color=INK2, lw=1)
    ax.set_yticks(y)
    ax.set_yticklabels([names[c] for c in order])
    ax.grid(axis="y", visible=False)
    ax.set_xlim(-0.42, 0.58)
    ax.set_xlabel("Share of index weight (sign = direction of effect)")
    ax.legend(loc="lower center", bbox_to_anchor=(0.4, 1.0), ncol=2)
    title(fig, "What the data says the MEI should weigh", "For defensive linemen, acceleration dominates and smoothness counts against.")
    save(fig, "fig5_weights.png")


def main():
    fig_coverage()
    fig_transfer()
    fig_smoothness()
    fig_models()
    fig_weights()


if __name__ == "__main__":
    main()


In [ ]:
%%writefile tests/__init__.py
"""Unit tests."""


In [ ]:
%%writefile tests/test_kinematics.py
import numpy as np
import pytest

from src.features.kinematics import DT, attempt_features


def sprint(vmax=10.0, tau=0.8, secs=5.0):
    t = np.arange(0, secs, DT)
    v = vmax * (1 - np.exp(-t / tau))
    x = np.cumsum(v) * DT + 10
    return x, np.zeros_like(x), v


def cut_run(turn_deg=90.0, v0=7.0, vmin=2.5, left=True):
    """Run straight, slow into a cut, turn, re-accelerate."""
    speeds = np.r_[np.linspace(0, v0, 15), np.full(10, v0), np.linspace(v0, vmin, 8),
                   np.full(3, vmin), np.linspace(vmin, v0, 12), np.full(10, v0)]
    turn = np.radians(turn_deg) * (1 if left else -1)
    heading = np.r_[np.zeros(30), np.linspace(0, turn, 8), np.full(len(speeds) - 38, turn)]
    x = np.cumsum(speeds * np.cos(heading)) * DT
    y = np.cumsum(speeds * np.sin(heading)) * DT
    return x, y, speeds


def test_straight_sprint_has_no_cuts():
    f = attempt_features(*sprint())
    assert f.cuts == []
    assert f.peak_speed == pytest.approx(10.0, rel=0.03)


def test_faster_starter_has_higher_accel_eff():
    quick = attempt_features(*sprint(tau=0.5))
    slow = attempt_features(*sprint(tau=1.2))
    assert quick.accel_eff > slow.accel_eff


def test_cut_detected_with_turn_size_and_sign():
    left = attempt_features(*cut_run(left=True))
    right = attempt_features(*cut_run(left=False))
    assert len(left.cuts) == 1 and len(right.cuts) == 1
    assert left.cuts[0].turn == pytest.approx(90, abs=15)
    assert right.cuts[0].turn == pytest.approx(-90, abs=15)


def test_harder_stop_scores_higher_decel_control():
    sharp = attempt_features(*cut_run(vmin=1.5))
    soft = attempt_features(*cut_run(vmin=4.5))
    assert sharp.decel_ctrl > soft.decel_ctrl


def test_smaller_speed_loss_means_more_efficient_cut():
    efficient = attempt_features(*cut_run(vmin=5.0))
    wasteful = attempt_features(*cut_run(vmin=2.0))
    assert efficient.cuts[0].cod_eff > wasteful.cuts[0].cod_eff


def test_noise_reduces_smoothness():
    x, y, v = sprint()
    rng = np.random.default_rng(0)
    noisy = attempt_features(x, y, v + rng.normal(0, 0.4, len(v)))
    assert attempt_features(x, y, v).smoothness > noisy.smoothness


def test_plant_and_reverse_detected():
    v = np.r_[np.linspace(0, 6, 10), np.linspace(6, 0.2, 6), np.linspace(0.2, 6, 10), np.full(10, 6.0)]
    sign = np.r_[np.ones(16), -np.ones(20)]
    x = np.cumsum(v * sign) * DT
    f = attempt_features(x, np.zeros_like(x), v)
    assert len(f.cuts) == 1
    assert abs(f.cuts[0].turn) == pytest.approx(180, abs=10)


def test_stationary_attempt_returns_none():
    z = np.zeros(30)
    assert attempt_features(z, z, z) is None


## 1. Unit tests for the movement metrics
Synthetic sprints, cuts and plant-and-reverse moves with known answers.

In [ ]:
!python -m pytest -q tests

## 2. Combine movement features and the MEI

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
pd.set_option('display.width', 200, 'display.max_columns', 30)

from src.features import combine
combine.main()
pd.read_parquet(combine.OUT / 'combine_features.parquet').describe().round(2)

In [ ]:
from src import analysis
analysis.validation().round(2)

## 3. NFL regular-season outcomes
Includes a pass over ~2 GB of game tracking for DB man-coverage throws.

In [ ]:
from src import outcomes
outcomes.main()

In [ ]:
analysis.reliability().round(2)

## 4. Models: traditional vs. tracking vs. hybrid
Ridge regression, leave-one-draft-class-out CV, 1,000 bootstrap resamples, 200 permutations.

In [ ]:
import sys
from src import models
sys.argv = ['models', '--perms', os.environ.get('BDB_PERMS', '200')]
models.main()
res = pd.read_csv(models.PROC / 'model_results.csv')
res[~res.model.str.startswith('Δ')].pivot_table(index=['position', 'outcome'], columns='model', values='rho').round(2)

## 5. Transfer test, learned weights, coverage

In [ ]:
analysis.main()

## 6. Figures

In [ ]:
from IPython.display import Image, display
from src import figures
figures.main()
for f in sorted(figures.FIG.glob('fig*.png')):
    display(Image(filename=str(f), width=900))